# BÖLGE BAZLI RANDOM FOREST

Eski kurulumun iki temel sorunu vardı:
- **Veri azlığı:** tüm depremlerin haftalık/yıllık ortalaması alınınca 75 bin kayıt 42 satıra iniyordu.
- **Ortalamanın sinyali yok etmesi:** küresel ortalama büyüklük neredeyse sabit, ortalama enlem/boylam ise gerçek bir yer değil.

Burada dünya ızgara hücrelerine bölünüyor ve her hücre ayrı bir zaman serisi oluyor (ayrıntı: `regional.py`).
- **Satır:** (hücre, dönem)
- **Özellikler:** son dönemlerdeki deprem sayısı, ortalama ve en büyük büyüklük, ortalama derinlik; hücrenin koordinatı ve o ana kadarki geçmiş ortalamaları
- **Hedef:** aynı hücrede bir sonraki dönemin **en büyük** depremi

Hızlı çalışması için grid küçük tutuldu (16 kombinasyon).

In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from regional import (build_panel, make_features, feature_columns, split_by_period, PeriodSplit,
                      LagSelector, evaluate_regional, TARGET_COL)

C:\Users\eren1\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
def run_rf(df, name, cell_deg, freq, max_lags, lag_options, model_path):
    panel = build_panel(df, cell_deg=cell_deg, freq=freq, min_active=0.8)
    data = make_features(panel, max_lags=max_lags)
    train, test = split_by_period(data)   # kronolojik: ilk %80 dönem train, son %20 test
    features = feature_columns(train)
    print(f"{name}: {panel.groupby(['cell_lat', 'cell_lon']).ngroups} hücre, {panel['period'].nunique()} dönem")

    search = GridSearchCV(
        Pipeline([('lags', LagSelector()), ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1))]),
        {'lags__n_lags': lag_options,
         'model__max_depth': [6, None],
         'model__min_samples_leaf': [5, 20],
         'model__max_features': [0.5, 1.0]},
        cv=PeriodSplit(n_splits=3),   # validation her zaman train'den sonraki dönemler
        scoring='neg_mean_squared_error', error_score='raise', n_jobs=-1)
    search.fit(train[features], train[TARGET_COL], groups=train['period_idx'])
    print(f"En iyi parametreler: {search.best_params_}, CV MSE: {-search.best_score_:.4f}")

    predictions = {'Random Forest': (search.predict(train[features]), search.predict(test[features]))}
    table, results = evaluate_regional(predictions, train, test, name)
    joblib.dump(search.best_estimator_, model_path)
    return search, table

## DATASET 1 (USGS, 2022): 1°×1° hücre × hafta

In [3]:
df1 = pd.read_csv('usgs_main.csv')
df1['time'] = pd.to_datetime(df1['time'])
df1 = df1[df1['type'] == 'earthquake']   # taş ocağı patlaması vb. kayıtlar deprem değil
search_1, table_1 = run_rf(df1, 'Bolgesel RF Dataset1', cell_deg=1.0, freq='W', max_lags=8,
                           lag_options=[2, 8], model_path='models/regional_rf_dataset1.pkl')

Bolgesel RF Dataset1: 110 hücre, 40 dönem


En iyi parametreler: {'lags__n_lags': 2, 'model__max_depth': 6, 'model__max_features': 0.5, 'model__min_samples_leaf': 20}, CV MSE: 0.4086

=== Bolgesel RF Dataset1 | 110 hücre, train: 2604 satır, test: 729 satır ===
Eşikler (train hedef dağılımı): %75 = 2.68, %90 = 3.38
                  Train R²  Test R²  Test MAE  Test RMSE  AUC (≥%75)  AUC (≥%90)
Model                                                                           
Persistence            NaN    0.013     0.608      0.817       0.744       0.777
Global ortalama        NaN   -0.002     0.624      0.823       0.500       0.500
Hücre ortalaması       NaN    0.408     0.464      0.633       0.767       0.814
Random Forest          0.5    0.429     0.455      0.622       0.766       0.801


## DATASET 2 (M≥5.5, küresel): 10°×10° hücre × yıl
1973 sonrası kullanılıyor; öncesinde yıllık kayıt sayısı çok daha az olduğu için yıllık deprem sayıları ve en büyük değerler dönemler arasında karşılaştırılabilir değil.

In [4]:
df2 = pd.read_csv('Significant Earthquake Dataset 1900-2023.csv')
df2 = df2.rename(columns={'Time':'time','Mag':'mag','Depth':'depth','Latitude':'latitude','Longitude':'longitude'})
df2['time'] = pd.to_datetime(df2['time'])
df2 = df2[df2['time'].dt.year >= 1973]
search_2, table_2 = run_rf(df2, 'Bolgesel RF Dataset2', cell_deg=10.0, freq='Y', max_lags=6,
                           lag_options=[2, 6], model_path='models/regional_rf_dataset2.pkl')

Bolgesel RF Dataset2: 48 hücre, 49 dönem


En iyi parametreler: {'lags__n_lags': 6, 'model__max_depth': None, 'model__max_features': 0.5, 'model__min_samples_leaf': 20}, CV MSE: 0.3043

=== Bolgesel RF Dataset2 | 48 hücre, train: 1569 satır, test: 410 satır ===
Eşikler (train hedef dağılımı): %75 = 6.80, %90 = 7.20
                  Train R²  Test R²  Test MAE  Test RMSE  AUC (≥%75)  AUC (≥%90)
Model                                                                           
Persistence            NaN   -0.646     0.595      0.771       0.572       0.560
Global ortalama        NaN   -0.000     0.480      0.601       0.500       0.500
Hücre ortalaması       NaN    0.113     0.442      0.566       0.663       0.607
Random Forest        0.363    0.123     0.444      0.563       0.678       0.615
